In [3]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    # device_map="cpu",
    # device_map="cuda:0",
    device_map = "mps",
    # device_map = "mlx-tune",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
    attn_implementation="sdpa",    # Use SDPA instead of FlashAttention2
)

ref_audio = "/Users/lorenzgutscher/Projects/Qwen3-TTS/hoi_goi_goi_051.wav"
ref_text  = "Du bist jo nid grouß gnuag um a flaschn Wein ausztrinka."

wavs, sr = model.generate_voice_clone(
    text="Gib dem Pepi sei Handy zrück!",
    language="German",
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write("output_voice_clone.wav", wavs[0], sr)

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 1587.85it/s]
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [ ]:
import json
from pathlib import Path
src = Path('/Users/lorenzgutscher/Projects/Qwen3-TTS/data/train-text-pac_orig_csc.txt')
out = Path('/Users/lorenzgutscher/Projects/Qwen3-TTS/data/train_raw_csc.jsonl')
ref_audio = './data/csc_ivg/csc_ivg_ivg_001.wav'
lines = []
for raw in src.read_text(encoding='utf-8').splitlines():
    raw = raw.strip()
    if not raw:
        continue
    key, text = raw.split('|', 1)
    obj = {"audio": f'./data/csc_ivg/{key}.wav', "text": text, "ref_audio": ref_audio}
    lines.append(json.dumps(obj, ensure_ascii=False, separators=(',', ':')))
out.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Wrote {len(lines)} compact JSONL rows to {out}')
print(out.read_text(encoding='utf-8').splitlines()[0])

Wrote /Users/lorenzgutscher/Projects/Qwen3-TTS/data/train_raw_csc.json with 621 lines
{"audio": "./data/csc_ivg/csc_ivg_ivg_002.wav", "text": "Ihr fahrt morgen nach Sillian.", "ref_audio": "./data/csc_ivg/csc_ivg_ivg_001.wav"}


In [ ]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

text1 = "Wir sagen zu der Maame und zum Fota des."
text1 = "Wia sogn zu da mame und zum Fota des."
text2 = "Hast du die Kühe schon gemolken?"
text3 = "miɐsɔɡŋtsɐdɐmamɛuntsʏmfɔːdɐðɛes"
text = text1
epoch = "1"

device = "mps"
tts = Qwen3TTSModel.from_pretrained(
    f"finetuning/output/checkpoint-epoch-{epoch}",
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",    # Use SDPA instead of FlashAttention2
    )

wavs, sr = tts.generate_custom_voice(
    text=text,
    speaker="csc",
)
sf.write(f"output_csc_test_update_epoch_{epoch}.wav", wavs[0], sr)
# sf.write(f"output_csc_test_phoneinput_epoch_{epoch}.wav", wavs[0], sr)



In [34]:
# text1 = "Wia sogn zu da mame und zum Fota des."

tokenizer = tts.processor.tokenizer
tokens = tokenizer.encode(text1)
print(tokens)
print(tokenizer.convert_ids_to_tokens(tokens))


[54, 685, 274, 3934, 6395, 2994, 296, 373, 2030, 16034, 434, 6089, 939, 13]
['W', 'ia', 'Ġs', 'ogn', 'Ġzu', 'Ġda', 'Ġm', 'ame', 'Ġund', 'Ġzum', 'ĠF', 'ota', 'Ġdes', '.']
